# RAG-Powered Knowledge Assistant

This notebook presents a polished implementation of a Retrieval-Augmented Generation knowledge assistant, designed for software portfolio presentation. The system combines semantic retrieval and generative modeling to answer queries with accurate, context-aware responses from a document corpus.

Core technology stack:
- ChromaDB for vector indexing and semantic search
- Hugging Face Transformers for encoder and generation models
- PyTorch as the model runtime and training framework


## Technical Architecture

The solution is built around a modular RAG pipeline:
- Documents are split into configurable chunks with overlap to optimize retrieval coverage.
- Each chunk is encoded into a vector representation and stored in a ChromaDB collection with metadata.
- User queries are encoded and used to retrieve the most relevant chunks from the vector store.
- Retrieved context is combined with the query and fed into a generative model to produce answers grounded in the knowledge base.
- Responses include traceable source references and support confidence-based handling for out-of-scope queries.


## Implementation Overview

This project includes a complete RAG pipeline and supporting infrastructure:

- Ingestion pipeline for structured document import, text normalization, and chunk generation.
- Embedding workflow using SentenceTransformer models to convert text chunks into dense vectors.
- ChromaDB vector store setup with metadata tagging for efficient retrieval.
- Query handling that performs semantic search, ranks candidates, and constructs a retrieval context.
- Generative response flow that merges retrieved content with the user query for grounded answer generation.
- Output handling that includes source attribution and confidence indicators for improved traceability.

---

## Part 1: Setup and Configuration

Let's start by installing and importing the required libraries.

In [ ]:
# Install required libraries
%pip install transformers torch sentence-transformers chromadb --quiet

print("Installation complete")

In [ ]:
# Import all required libraries
import chromadb
from chromadb.utils import embedding_functions
from sentence_transformers import SentenceTransformer, util
from transformers import pipeline
import torch
import re
from typing import List, Dict, Optional
import json

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print("\n All imports successful!")

---

## Part 2: Project Declaration

Before building, declare your project details. This helps you plan and will be part of your submission.

In [ ]:
# ============================================================
# TODO: Customer support agent
# ============================================================

PROJECT_INFO = {
    "scenario": "Customer Support",
    "topic": "AlloyCut Supply Co. (Industrial Metal & Precision Cutting)",
    "description": "An intelligent support assistant designed to handle B2B client inquiries regarding raw metal inventories, precision custom cutting tolerances, wholesale tiered pricing structures, shipping fulfillment workflows, and return policies.",
    "target_users": "Procurement managers, quality control inspectors, CNC machine shops, and manufacturing fabricators looking for fast specifications and pricing updates.",
}

# Print your declaration
print("🚀 PROJECT DECLARATION")
print("=" * 50)
for key, value in PROJECT_INFO.items():
    print(f"{key.replace('_', ' ').title()}: {value}")

---

## Part 3: Create Your Knowledge Base

This is the foundation of your RAG system. Create comprehensive, well-organized documents for your chosen scenario.

### Guidelines for Good Knowledge Base Documents:
- **Be specific**: Include names, numbers, dates, and concrete details
- **Be comprehensive**: Cover the topic thoroughly
- **Be organized**: Structure information logically
- **Be accurate**: Ensure facts are correct and consistent across documents

In [ ]:
# ============================================================
# TODO: Create your knowledge base with at least 8 documents
# ============================================================

# Each document should have:
# - A clear, descriptive key (used as document ID)
# - Content of at least 150 words
# - Specific, factual information

knowledge_base = {
    "document_1": """
    [Company Overview and Core Catalog Products]
    AlloyCut Supply Co. is a leading national distributor and precision processing center for industrial metals, servicing high-precision manufacturing, aerospace engineering, and automotive fabrication clients. Founded in 2014, our primary warehouse inventory features premium-grade metals categorized into three core product families. Our first product family is Aerospace-Grade Aluminum, featuring 6061-T6 and 7075-T6 alloy plates, sheets, and extruded bars ideal for structural components. Our second product family is Structural and Stainless Steel, which includes 304, 316, and 4130 chromoly steel alloys widely utilized for heavy machinery, structural frames, and corrosive environments. Our third product family is Industrial Copper and Brass Alloys, featuring C360 Free-Cutting Brass and C110 ETP Copper bars optimized for electrical conductivity and precision custom machining. All materials stocked by AlloyCut Supply Co. are fully traceably sourced from certified domestic mills, accompanied by Mill Test Reports (MTRs) to guarantee mechanical compliance. We ensure that every shipment undergoes rigorous inbound quality assurance inspections to verify chemistry and structural integrity before being placed into our live inventory tracking system.
    """,

    "document_2": """
    [Precision Processing and Cutting Services]
    Beyond raw material distribution, AlloyCut Supply Co. provides advanced, high-tolerance precision processing and custom cutting services directly at our fulfillment center. Our processing facility is equipped with industrial-grade machinery, including the Apex-3000 CNC Waterjet System, the Vulcan Laser Cutter, and several high-capacity Hyd-Mech automatic bandsaws. We offer custom linear saw cutting for all metal bars and tubes with a standard operational tolerance of +/- 0.030 inches. For high-precision sheets and complex geometries, our CNC Waterjet processing delivers an exceptional edge finish with an elite tolerance of +/- 0.005 inches, completely eliminating thermal heat-affected zones that alter metal temper. Our Vulcan Laser cutting service is reserved for carbon and stainless steel sheets up to 0.500 inches thick, achieving processing speeds up to four times faster than traditional methods with a precision tolerance of +/- 0.010 inches. Clients requesting custom processing must submit structured engineering blueprints or standard CAD files (.DXF or .DWG format) directly to our engineering queue. All custom-cut orders undergo three-dimensional coordinate measuring machine (CMM) verification by our quality control inspectors to guarantee exact conformance to specified dimensional tolerances before shipping.
    """,

    "document_3": """
    [Pricing, Accounts, and Tiered Wholesale Discount Plans]
    AlloyCut Supply Co. utilizes a dynamic, volume-based tiered pricing matrix designed to offer substantial cost savings to high-volume manufacturing partners and repeat industrial buyers. Category 1 is Tier 1 Retail Pricing, which applies to all standard orders with a total material weight under 500 pounds, calculated at baseline market spot prices per pound plus a flat processing fee of $45 for custom cuts. Category 2 is Tier 2 Preferred Commercial Pricing, automatically applied to single-order volumes between 500 and 2,499 pounds, granting a 7.5% discount on raw material costs and reducing processing fees to a flat $20. Category 3 is Tier 3 Enterprise Wholesale Pricing, triggered by order volumes exceeding 2,500 pounds or an annual contract commitment of 20 tons, providing a 15% discount on raw materials and completely waiving all custom cutting setup charges. Corporate clients looking to optimize cash flow can apply for formal commercial credit lines through our corporate finance office. We offer standard Net-30 and Net-60 payment terms upon a comprehensive credit review, which typically takes three to five business days. Approved credit accounts receive a dedicated account manager and gain instant access to our automated online customer portal for real-time freight tracking and instant quote generation.
    """,

    "document_4": """
    [Fulfillment, Shipping, and Freight Delivery Logistics]
    Efficient logistics management is central to operations at AlloyCut Supply Co., ensuring that raw materials arrive safely at manufacturing plants without disrupting production schedules. Standard in-stock orders of raw metal bars and sheets feature a processing lead time of 24 to 48 hours before leaving our loading docks. Custom processing orders requiring CNC waterjet or laser cutting typically carry an extended production lead time of 3 to 5 business days, depending on engineering queue volume. Local shipments within a 150-mile radius of our primary Midwest hub are delivered directly via our private fleet of AlloyCut flatbed trucks, carrying a flat delivery fee of $85 per drop-off. For national orders, we partner exclusively with certified Less-Than-Truckload (LTL) freight carriers, including Old Dominion Freight Line and FedEx Freight, to transport heavy pallets safely. Standard national LTL freight shipping typically delivers within 3 to 7 business days depending on the destination region. Expedited hot-shot freight delivery is available upon request for critical plant shutdown situations, guaranteeing transit times within 24 to 48 hours nationwide for an additional premium logistics fee calculated at time of booking.
    """,

    "document_5": """
    [Order Modifications, Cancellations, and Adjustments Policy]
    AlloyCut Supply Co. maintains a strict administrative policy regarding order modifications and cancellations to maintain high operational efficiency across our automated warehouse floors. Customers can modify or cancel any standard, stock-length order without penalty within 4 hours of receiving their initial order confirmation email by contacting support. Once an order passes this 4-hour window, it enters active warehouse picking and staging, incurring a mandatory 15% restocking fee if cancelled. Crucially, orders involving custom processing, custom linear bandsaw cutting, CNC waterjet shaping, or specialized laser profiling cannot be modified or cancelled under any circumstances once the engineering team approves the CAD blueprint and releases the job to the shop floor. If an administrative data entry error is discovered by the client after submission, they must immediately contact the Order Processing Desk at extension 104. In cases where the material has not yet physically loaded onto a cutting bed, our shop foreman can pause production to apply corrections, though this may incur an administrative adjustment fee of $50 to cover updated material tracking sheets.
    """,

    "document_6": """
    [Comprehensive Returns, Material Discrepancies, and Refunds Policy]
    Customer satisfaction and material integrity are paramount at AlloyCut Supply Co., which is why we enforce an explicit 30-day Return and Material Discrepancy Policy. Standard, unaltered stock metal lengths can be returned within 30 calendar days of delivery, provided the metal remains in its original, unmarred condition with the mill heat numbers and material labels perfectly legible. All approved standard returns are subject to a 20% restocking fee to cover inbound inspection and warehousing costs, and the customer is responsible for arranging freight shipping back to our hub. Custom-processed, waterjet-cut, or laser-profiled materials are completely non-returnable and non-refundable, as these items are tailored to unique customer blueprints. However, if a customer receives material that deviates from their engineering blueprint, or if the metal exhibits structural defects, the customer must file a formal Quality Non-Conformance Claim within 10 business days of delivery. Claims must include clear photographs of the dimensional deviation alongside digital caliper or micrometer readings sent to our quality desk. If our inspection team confirms a processing error exceeding specified tolerances, AlloyCut will issue a full 100% refund or expedite a replacement order at zero cost.
    """,

    "document_7": """
    [Troubleshooting Common Material Defects and Metal Handling Guides]
    Proper material storage and handling are critical to preventing surface defects and maintaining structural properties prior to manufacturing operations. A common customer troubleshooting issue involves surface oxidation, frequently observed as white rust on aluminum sheets or light red scaling on structural carbon steel. This is typically caused by improper storage in high-humidity environments or exposure to condensation during rapid temperature swings. To prevent or mitigate surface oxidation, AlloyCut Supply Co. strongly recommends storing all raw metal stock indoors on elevated racks away from direct contact with concrete floors, which leach moisture. If surface rust has already formed on carbon steel, it can safely be removed using a wire wheel brush or a mild chemical rust converter without compromising the underlying structural integrity of the steel. Another frequent issue is tool chatter and premature bit wear during the machining of stainless steel alloys like 316. This issue occurs when CNC operators run spindle speeds too high, causing the metal to work-harden rapidly. This can be successfully resolved by reducing cutting speeds, increasing feed rates, and ensuring a continuous flow of water-soluble cooling lubricant directly to the cutting edge to dissipate thermal energy.
    """,

    "document_8": """
    [Corporate Contact Information, Working Hours, and Help Desk Directory]
    AlloyCut Supply Co. is dedicated to providing elite customer service and responsive technical communication across all operational departments. Our corporate headquarters and central distribution facility are located at 4100 Industrial Parkway, Suite B, Cleveland, Ohio 44135. Our standard operating business hours are Monday through Friday, from 7:00 AM to 6:00 PM Eastern Standard Time (EST), while our warehouse loading docks close promptly at 5:00 PM EST for outbound freight carrier pickups. Customers looking for immediate assistance can reach our main corporate telephone hotline at 1-800-555-METL (6385) during standard business hours. For quick routing, dial Extension 101 for the General Sales and New Quotes Desk to get immediate material pricing. Dial Extension 102 for the Technical Engineering Team to discuss CAD file submissions and CNC processing requirements. Dial Extension 103 for the Shipping and Logistics Coordinator to check on pending LTL freight tracking and delivery windows. Finally, dial Extension 104 to reach the Quality Assurance and Non-Conformance Claims Desk to report material defects or filing returns. For digital inquiries outside of standard working hours, clients can email our unified support inbox at support@alloycutsupply.com, where our team guarantees a comprehensive response within 12 business hours.
    """
}

# Validation Code Block (Kept exactly as requested by your notebook)
print("🔍 KNOWLEDGE BASE VALIDATION")
print("=" * 50)
print(f"Total documents: {len(knowledge_base)}")

total_words = 0
for doc_name, content in knowledge_base.items():
    word_count = len(content.split())
    total_words += word_count
    status = "Pass" if word_count >= 150 else "Too short!"
    print(f"  {doc_name}: {word_count} words | Status: {status}")

print(f"\nTotal words: {total_words}")
print(f"Average words per document: {total_words // len(knowledge_base)}")

if len(knowledge_base) >= 8:
    print("\n✅ Document count requirement met!")
else:
    print(f"\n❌ You need at least 8 documents. Currently have: {len(knowledge_base)}")

---

## Part 4: Document Chunking

Implement a chunking function and process your knowledge base. You should experiment with chunk size to find what works best for your content.

In [ ]:
# ============================================================
# TODO: Implement your chunking function
# ============================================================

def chunk_document(
    text: str,
    chunk_size: int = 400,
    chunk_overlap: int = 50
) -> List[str]:
    """
    Split a document into overlapping chunks.

    Args:
        text: The document text to chunk
        chunk_size: Target size for each chunk (in characters)
        chunk_overlap: Number of characters to overlap between chunks

    Returns:
        List of text chunks

    Requirements:
        - Respect sentence boundaries (don't cut mid-sentence)
        - Include overlap between consecutive chunks
        - Handle edge cases (very short documents, etc.)
    """
    # Requirement 3: Handle edge cases (empty or very short documents)
    if not text.strip():
        return []
    if len(text) <= chunk_size:
        return [text.strip()]

    # Requirement 1: Split text into sentences using regex to respect boundaries
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    chunks = []
    current_chunk = ""

    for sentence in sentences:
        # If adding the next sentence exceeds chunk_size, we close the current chunk
        if len(current_chunk) + len(sentence) > chunk_size and current_chunk:
            chunks.append(current_chunk.strip())

            # Requirement 2: Include overlap from the end of the current chunk
            overlap_start = max(0, len(current_chunk) - chunk_overlap)
            overlap_text = current_chunk[overlap_start:]

            # Clean up the overlap start so it doesn't break in the middle of a word
            if " " in overlap_text:
                overlap_text = overlap_text[overlap_text.find(" "):]

            current_chunk = overlap_text.strip() + " " + sentence
        else:
            if current_chunk:
                current_chunk += " " + sentence
            else:
                current_chunk = sentence

    # Append the last remaining chunk
    if current_chunk.strip():
        chunks.append(current_chunk.strip())

    return chunks


# Test your chunking function
test_text = list(knowledge_base.values())[0]
test_chunks = chunk_document(test_text)

print("🔍 CHUNKING TEST")
print("=" * 50)
print(f"Original document length: {len(test_text)} characters")
print(f"Number of chunks created: {len(test_chunks)}")
print(f"\nFirst chunk preview:")
print(test_chunks[0][:200] + "..." if len(test_chunks) > 0 else "No chunks created")

## Chunking Strategy

For this project, I selected a **chunk size of 400 characters** with a **50-character overlap**. My knowledge base contains customer support documentation, including product information, pricing, troubleshooting guides, company policies, and contact information. These documents contain many independent sections with factual information, making a moderate chunk size ideal for semantic retrieval.

A chunk size of 400 characters provides enough context for the embedding model to understand the meaning of each section without including excessive unrelated information. Smaller chunks improve retrieval precision because the vector database can return more focused results that closely match the user's question.

I selected a 50-character overlap to preserve context between adjacent chunks. Important details such as pricing, warranty information, or troubleshooting instructions can occasionally span chunk boundaries. The overlap helps ensure that these details remain complete within at least one chunk, reducing the chance of missing relevant information during retrieval.

Each chunk is stored in ChromaDB with metadata that includes the source document, document title, and chunk index. This metadata allows the RAG system to provide source attribution and improves traceability when generating responses.

This chunking strategy provides a good balance between retrieval accuracy and contextual completeness while remaining flexible enough to adjust for larger or more technical knowledge bases in future implementations.

In [ ]:
# ============================================================
# TODO: Process all documents into chunks with metadata
# ============================================================

# Establish structural chunking parameters
CHUNK_SIZE = 400
CHUNK_OVERLAP = 50

# Initialize global arrays for vector database ingestion
all_chunks = []
all_metadatas = []
all_ids = []

# Process each document in the knowledge base
for doc_id, content in knowledge_base.items():
    # Parse out the clean document title from the structural header
    first_line = content.strip().split('\n')[0]
    title = first_line.replace('[', '').replace(']', '').strip()

    # Segment the text using the sentence-boundary chunking function
    chunks = chunk_document(content, chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)

    # Format and store every text segment with tracking metadata
    for idx, chunk in enumerate(chunks):
        # Generate a distinct identifier for database indexing
        unique_id = f"{doc_id}_chunk_{idx}"

        # Inject explicit document lineage directly into the text data
        contextualized_text = f"Document Title: {title}\nContent: {chunk}"

        # Populate tracking collections
        all_chunks.append(contextualized_text)
        all_metadatas.append({
            "source": doc_id,
            "title": title,
            "chunk_index": idx
        })
        all_ids.append(unique_id)

# Execute verification summary
print("CHUNKING RESULTS")
print("=" * 50)
print(f"Total chunks created: {len(all_chunks)}")
print(f"Chunk size setting: {CHUNK_SIZE}")
print(f"Chunk overlap setting: {CHUNK_OVERLAP}")

### Chunking Strategy Justification

**TODO:** In the cell below, explain your chunking decisions:

*Double-click to edit this cell*

**Why did you choose this chunk size?**

A target chunk size of 400 characters (roughly 60–80 words) was selected to balance retrieval precision and contextual density. Because we are using an extractive QA model (distilbert-base-cased-distilled-squad), keeping the chunks smaller ensures that the model isn't overwhelmed with irrelevant text, which maintains a higher signal-to-noise ratio during the semantic vector search phase.

**Why did you choose this overlap amount?**

An overlap of 50 characters ensures semantic continuity between adjacent text segments. Since the chunking algorithm splits text at sentence boundaries, this safety buffer guarantees that phrases or technical specifications (like dimensions or pricing categories) spanning across boundary definitions are not accidentally dropped or contextually isolated.

**Did you make any modifications to handle your specific content type?**

Yes. Since a customer support assistant requires high data integrity for names, metrics, and pricing, the chunk text was prepended with explicit document context. This structural modification anchors localized keyword metadata inside the embedded string space, ensuring that even if a sentence chunk contains generic language, its relational context back to the primary topic remains strong inside ChromaDB.

---

## Part 5: Vector Database Setup

Create your ChromaDB collection and add all chunks with embeddings.

In [ ]:
# ============================================================
# TODO: Set up ChromaDB and create your collection
# ============================================================

# Initialize ChromaDB client
chroma_client = chromadb.Client()

# Create embedding function using the standard sentence-transformer model
embedding_function = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

# Establish the collection dedicated to the alloycut business data
collection = chroma_client.create_collection(
    name="alloycut_support_knowledge_base",
    embedding_function=embedding_function,
    metadata={
        "description": "Vector store containing customer support records, technical specs, pricing matrices, and fulfillment logistics for AlloyCut Supply Co."
    }
)

print("Collection created!")

In [ ]:
# ============================================================
# TODO: Add all chunks to the collection
# ============================================================

# Ingest all formatted chunks, associated metadata, and unique identifiers
collection.add(
    documents=all_chunks,
    metadatas=all_metadatas,
    ids=all_ids
)

# Verify successful database insertion
print(f"✅ Added {collection.count()} chunks to the database")

In [ ]:
# Execute a test retrieval to verify semantic search alignment
test_query = "What is the tolerance for custom linear saw cutting?"

results = collection.query(
    query_texts=[test_query],
    n_results=3
)

# Print execution diagnostics
print(f"Test Query: \"{test_query}\"")
print("\nTop 3 Retrieved Chunks:")
for i in range(len(results['documents'][0])):
    print(f"\n{i+1}. Source: {results['metadatas'][0][i].get('source', 'Unknown')}")
    print(f"   Preview: {results['documents'][0][i][:150]}...")

---

## Part 6: Build the RAG Pipeline

Create a complete RAG pipeline class with all required functionality.

In [ ]:
# Load the question-answering model
# NOTE: Using direct AutoModel instantiation to bypass the environment's pipeline KeyError
import torch
from transformers import AutoModelForQuestionAnswering, AutoTokenizer

model_checkpoint = "distilbert-base-cased-distilled-squad"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
model = AutoModelForQuestionAnswering.from_pretrained(model_checkpoint)

def qa_model(question: str, context: str) -> dict[str, float | str]:
    """Run extractive QA to identify the most likely answer span in the provided context."""
    """
    The model outputs separate logits for span start and end positions. This function
    selects the highest-probability start and end tokens, decodes the corresponding
    token span, and converts the model logits into a normalized confidence score.

    Args:
        question: User query text.
        context: Retrieved document text that may contain the answer.

    Returns:
        A dictionary with the extracted answer string and a confidence score.
    """
    inputs = tokenizer(question, context, return_tensors="pt", truncation=True, max_length=512)
    with torch.no_grad():
        outputs = model(**inputs)

    start_logits = outputs.start_logits[0]
    end_logits = outputs.end_logits[0]

    start_idx = int(torch.argmax(start_logits).item())
    end_idx = int(torch.argmax(end_logits).item())

    if end_idx < start_idx:
        end_idx = start_idx

    start_probs = torch.softmax(start_logits, dim=-1)
    end_probs = torch.softmax(end_logits, dim=-1)

    start_score = float(start_probs[start_idx].item())
    end_score = float(end_probs[end_idx].item())
    confidence_score = (start_score + end_score) / 2.0

    input_ids = inputs["input_ids"][0]
    answer_tokens = input_ids[start_idx : end_idx + 1]
    answer_text = tokenizer.decode(answer_tokens, skip_special_tokens=True).strip()

    return {
        "answer": answer_text,
        "score": confidence_score
    }

# Assign to the notebook's expected variable name
generator = qa_model
print("QA model loaded!")

In [ ]:
class RAGAssistant:
    """Retrieval-augmented assistant with confidence-based response gating."""

    def __init__(self, collection, qa_model, confidence_threshold: float = 0.3):
        """Initialize the RAG assistant.

        Args:
            collection: ChromaDB collection containing embedded document chunks.
            qa_model: Callable extractive QA model that accepts question/context.
            confidence_threshold: Minimum fused confidence required to return
                a direct answer.
        """
        self.collection = collection
        self.qa_model = qa_model
        self.confidence_threshold = confidence_threshold

    def retrieve(self, query: str, n_results: int = 3) -> Dict:
        """Retrieve top matching chunks from the vector store.

        Args:
            query: User question used for semantic retrieval.
            n_results: Number of top results to return.

        Returns:
            Retrieval payload with documents, metadata, and vector distances.
        """
        return self.collection.query(
            query_texts=[query],
            n_results=n_results,
            include=["documents", "metadatas", "distances"],
        )

    def generate_answer(self, question: str, context: str) -> Dict:
        """Generate an extractive answer span from retrieved context.

        The underlying QA model predicts start and end token logits across the
        context sequence. The selected span is decoded as the candidate answer,
        and its confidence score is used for downstream gating.

        Args:
            question: Original user question.
            context: Concatenated retrieved text.

        Returns:
            Dictionary with keys:
                answer: Extracted answer text.
                confidence: Model-provided confidence score.
        """
        extraction = self.qa_model(question=question, context=context)
        answer = extraction.get("answer", "").strip()
        confidence = float(extraction.get("score", 0.0))

        invalid_span = (not answer) or (len(answer) <= 1) or (answer.lower() in question.lower())
        if invalid_span:
            return {
                "answer": "I do not have enough information in the knowledge base to answer that.",
                "confidence": 0.0,
            }

        return {"answer": answer, "confidence": confidence}

    def ask(self, question: str, n_results: int = 3) -> Dict:
        """Execute retrieval, answer generation, and confidence fusion.

        Args:
            question: User question.
            n_results: Number of chunks to retrieve.

        Returns:
            Dictionary containing question, final answer, confidence,
            sources, and confidence status.
        """
        fallback_answer = (
            "I'm sorry, but I do not have enough confident information in my knowledge "
            "base to accurately answer this question."
        )

        retrieval_results = self.retrieve(question, n_results=n_results)
        retrieved_docs = retrieval_results.get("documents", [[]])[0]
        retrieved_metadatas = retrieval_results.get("metadatas", [[]])[0]
        retrieved_distances = retrieval_results.get("distances", [[]])[0]

        if not retrieved_docs:
            return {
                "question": question,
                "answer": fallback_answer,
                "confidence": 0.0,
                "sources": [],
                "is_confident": False,
            }

        context = "\n\n".join(retrieved_docs)
        generated = self.generate_answer(question, context)

        avg_distance = (
            sum(retrieved_distances) / len(retrieved_distances)
            if retrieved_distances
            else 1.0
        )
        retrieval_confidence = max(0.0, 1.0 - avg_distance)
        generation_confidence = float(generated.get("confidence", 0.0))

        final_confidence = min(generation_confidence, retrieval_confidence)
        is_confident = final_confidence >= self.confidence_threshold

        sources: List[str] = []
        for meta in retrieved_metadatas:
            title = meta.get("title", meta.get("source", "Unknown Source"))
            if title not in sources:
                sources.append(title)

        final_answer = generated.get("answer", "").strip()
        if (not final_answer) or (not is_confident):
            final_answer = fallback_answer
            final_confidence = 0.0
            is_confident = False

        return {
            "question": question,
            "answer": final_answer,
            "confidence": final_confidence,
            "sources": sources,
            "is_confident": is_confident,
        }

    def format_response(self, result: Dict) -> str:
        """Format pipeline output for presentation.

        Args:
            result: Output dictionary returned by ask().

        Returns:
            Human-readable response with confidence and source attribution.
        """
        status = "CONFIDENT" if result["is_confident"] else "LOW_CONFIDENCE"
        response = f"[{status}] Question: {result['question']}\n\n"
        response += f"Answer: {result['answer']}\n\n"
        response += f"Confidence: {result['confidence']:.2%}\n"

        if result["sources"]:
            response += f"Sources: {', '.join(result['sources'])}"
        else:
            response += "Sources: None"

        return response


assistant = RAGAssistant(collection, generator, confidence_threshold=0.3)

print("RAG Assistant created.")

In [ ]:
# ============================================================
# Test your assistant with a few questions
# ============================================================

# Initialize evaluation array with specific customer support questions
test_questions = [
    "What is the corporate phone number for AlloyCut Supply Co.?",
    "What is the precision tolerance for CNC Waterjet processing?",
    "Do you sell medical-grade titanium sheets or custom alloy fasteners?",
]

print("INITIAL TESTING")
print("=" * 60)
for question in test_questions:
    result = assistant.ask(question)
    print(f"\nQ: {question}")
    print(assistant.format_response(result))

---

## Part 7: Comprehensive Testing

Test your system thoroughly with at least 15 questions covering different types and difficulty levels.

In [ ]:
class RAGAssistant:
    """Retrieval-augmented assistant with confidence-based response gating.

    This assistant retrieves semantically relevant context from ChromaDB,
    runs extractive QA over the retrieved context, and applies reliability
    guardrails before returning a final response.
    """

    def __init__(self, collection, qa_model, confidence_threshold: float = 0.3):
        """Initialize the assistant.

        Args:
            collection: ChromaDB collection containing embedded document chunks.
            qa_model: Callable extractive QA model that accepts question/context.
            confidence_threshold: Minimum fused confidence required to return
                a direct answer.
        """
        self.collection = collection
        self.qa_model = qa_model
        self.confidence_threshold = confidence_threshold

    def retrieve(self, query: str, n_results: int = 3) -> Dict:
        """Retrieve top matching chunks from the vector store.

        Args:
            query: User question used for semantic retrieval.
            n_results: Number of top results to return.

        Returns:
            Retrieval payload with documents, metadata, and vector distances.
        """
        return self.collection.query(
            query_texts=[query],
            n_results=n_results,
            include=["documents", "metadatas", "distances"],
        )

    def generate_answer(self, question: str, context: str) -> Dict:
        """Generate an extractive answer span from retrieved context.

        The underlying QA model predicts start and end token logits across the
        context sequence. The selected span is decoded as the candidate answer,
        and its confidence score is used for downstream gating.

        Args:
            question: Original user question.
            context: Concatenated retrieved text.

        Returns:
            Dictionary with keys:
                answer: Extracted answer text.
                confidence: Model-provided confidence score.
        """
        extraction = self.qa_model(question=question, context=context)
        answer = extraction.get("answer", "").strip()
        confidence = float(extraction.get("score", 0.0))

        # Reject degenerate spans that usually indicate insufficient grounding.
        invalid_span = (not answer) or (len(answer) <= 1) or (answer.lower() in question.lower())
        if invalid_span:
            return {
                "answer": "I do not have enough information in the knowledge base to answer that.",
                "confidence": 0.0,
            }

        return {"answer": answer, "confidence": confidence}

    def ask(self, question: str, n_results: int = 3) -> Dict:
        """Execute retrieval, answer generation, and confidence fusion.

        Args:
            question: User question.
            n_results: Number of chunks to retrieve.

        Returns:
            Dictionary containing question, final answer, confidence,
            sources, and confidence status.
        """
        fallback_answer = (
            "I'm sorry, but I do not have enough confident information in my knowledge "
            "base to accurately answer this question."
        )

        retrieval_results = self.retrieve(question, n_results=n_results)
        retrieved_docs = retrieval_results.get("documents", [[]])[0]
        retrieved_metadatas = retrieval_results.get("metadatas", [[]])[0]
        retrieved_distances = retrieval_results.get("distances", [[]])[0]

        if not retrieved_docs:
            return {
                "question": question,
                "answer": fallback_answer,
                "confidence": 0.0,
                "sources": [],
                "is_confident": False,
            }

        context = "\n\n".join(retrieved_docs)
        generated = self.generate_answer(question, context)

        avg_distance = (
            sum(retrieved_distances) / len(retrieved_distances)
            if retrieved_distances
            else 1.0
        )
        retrieval_confidence = max(0.0, 1.0 - avg_distance)
        generation_confidence = float(generated.get("confidence", 0.0))

        final_confidence = min(generation_confidence, retrieval_confidence)
        is_confident = final_confidence >= self.confidence_threshold

        sources: List[str] = []
        for meta in retrieved_metadatas:
            title = meta.get("title", meta.get("source", "Unknown Source"))
            if title not in sources:
                sources.append(title)

        final_answer = generated.get("answer", "").strip()
        if (not final_answer) or (not is_confident):
            final_answer = fallback_answer
            final_confidence = 0.0
            is_confident = False

        return {
            "question": question,
            "answer": final_answer,
            "confidence": final_confidence,
            "sources": sources,
            "is_confident": is_confident,
        }

    def format_response(self, result: Dict) -> str:
        """Format pipeline output for presentation.

        Args:
            result: Output dictionary returned by ask().

        Returns:
            Human-readable response with confidence and source attribution.
        """
        status = "CONFIDENT" if result["is_confident"] else "LOW_CONFIDENCE"
        response = f"[{status}] Question: {result['question']}\n\n"
        response += f"Answer: {result['answer']}\n\n"
        response += f"Confidence: {result['confidence']:.2%}\n"

        if result["sources"]:
            response += f"Sources: {', '.join(result['sources'])}"
        else:
            response += "Sources: None"

        return response


assistant = RAGAssistant(collection, generator, confidence_threshold=0.3)

print("RAG Assistant created.")

In [ ]:
# ============================================================
# Run comprehensive tests and collect results
# ============================================================

from textwrap import shorten

# Display configuration kept explicit for reproducible, reviewer-friendly output.
DISPLAY = {
    "table_width": 144,
    "category_width": 22,
    "question_width": 56,
    "answer_width": 42,
    "sources_width": 34,
}

def _clip(text: str, width: int) -> str:
    """Trim long text consistently so row alignment is preserved."""
    return shorten(" ".join(str(text).split()), width=width, placeholder="...")

def _sources_label(sources: list[str], max_sources: int = 2) -> str:
    """Create a compact source label for dashboard rendering."""
    if not sources:
        return "None"
    displayed = ", ".join(sources[:max_sources])
    suffix = "" if len(sources) <= max_sources else f" (+{len(sources) - max_sources} more)"
    return displayed + suffix

def _print_section(title: str, width: int) -> None:
    """Print a visual section divider used across categories."""
    rule = "=" * width
    print(f"\n{rule}")
    print(f" {title}")
    print(rule)

# Provide a safe default suite if the dedicated test suite cell was not run.
existing_test_suite = globals().get("test_suite")
if isinstance(existing_test_suite, dict) and existing_test_suite:
    test_suite = existing_test_suite
else:
    test_suite = {
        "simple_facts": [
            "Where is the corporate headquarters located?",
            "What is the standard delivery fee for a local shipment?",
            "What are the three core product families featured in the warehouse inventory?",
        ],
        "detailed_explanations": [
            "How can a customer safely remove surface rust if it has already formed on carbon steel?",
            "What is the administrative policy regarding order modifications within the four-hour window?",
            "What are the processing and production lead times required for orders that involve custom cutting services?",
        ],
        "comparison_or_complex": [
            "What is the difference in raw material pricing discounts between Tier 2 Preferred Commercial and Tier 3 Enterprise Wholesale plans?",
            "How does the standard operational tolerance for custom linear saw cutting compare to the precision tolerance of the Vulcan Laser cutting service?",
        ],
        "edge_cases": [
            "Do you sell medical-grade titanium sheets or custom alloy fasteners?",
            "What is your refund policy for heavy-duty copper cookware or kitchen backsplashes?",
            "Can I request structural carbon steel beams delivered directly to a residential address in Seattle, Washington?",
        ],
    }
    print("Using default test_suite because no prior test suite definition was found.")

test_results = []

_print_section("RAG EVALUATION DASHBOARD", DISPLAY["table_width"])
print(
    f" {'#':<3}"
    f" {'Status':<7}"
    f" {'Category':<{DISPLAY['category_width']}}"
    f" {'Confidence':>10}"
    f" {'Question':<{DISPLAY['question_width']}}"
    f" {'Answer Preview':<{DISPLAY['answer_width']}}"
    f" {'Sources':<{DISPLAY['sources_width']}}"
)
print("-" * DISPLAY["table_width"])

row_num = 0
for category, questions in test_suite.items():
    category_label = category.upper().replace("_", " ")
    _print_section(f"CATEGORY: {category_label}", DISPLAY["table_width"])

    for question in questions:
        row_num += 1
        result = assistant.ask(question)

        # Store structured results for downstream analysis.
        test_results.append(
            {
                "category": category,
                "question": question,
                "answer": result["answer"],
                "confidence": result["confidence"],
                "is_confident": result["is_confident"],
                "sources": result["sources"],
            }
        )

        status = "PASS" if result["is_confident"] else "REVIEW"
        confidence_pct = f"{result['confidence']:.2%}"
        question_preview = _clip(question, DISPLAY["question_width"])
        answer_preview = _clip(result["answer"], DISPLAY["answer_width"])
        sources_preview = _clip(_sources_label(result["sources"]), DISPLAY["sources_width"])

        print(
            f" {row_num:<3}"
            f" {status:<7}"
            f" {category_label:<{DISPLAY['category_width']}}"
            f" {confidence_pct:>10}"
            f" {question_preview:<{DISPLAY['question_width']}}"
            f" {answer_preview:<{DISPLAY['answer_width']}}"
            f" {sources_preview:<{DISPLAY['sources_width']}}"
        )

print("\n" + "-" * DISPLAY["table_width"])
print(f"Completed evaluation rows: {len(test_results)}")

In [ ]:
# ============================================================
# Analyze test results
# ============================================================

if not test_results:
    raise ValueError("No test results found. Run the evaluation loop cell first.")

SUMMARY_WIDTH = 86

def _summary_rule(char: str = "=") -> str:
    """Return a reusable horizontal divider for summary output."""
    return char * SUMMARY_WIDTH

def _fmt_pct(value: float) -> str:
    """Format fractional values as percentages with consistent precision."""
    return f"{value:.2%}"

def _print_kv(label: str, value: str) -> None:
    """Render one key-value line with aligned labels."""
    print(f" {label:<28}: {value}")

print("\n" + _summary_rule("="))
print(" RAG EVALUATION SUMMARY")
print(_summary_rule("="))

total_questions = len(test_results)
confident_answers = sum(1 for r in test_results if r["is_confident"])
review_answers = total_questions - confident_answers
avg_confidence = sum(r["confidence"] for r in test_results) / total_questions
pass_rate = confident_answers / total_questions

_print_kv("Total Questions", str(total_questions))
_print_kv("PASS (Confident)", str(confident_answers))
_print_kv("REVIEW (Low Confidence)", str(review_answers))
_print_kv("Pass Rate", _fmt_pct(pass_rate))
_print_kv("Average Confidence", _fmt_pct(avg_confidence))

print("\n" + _summary_rule("-"))
print(
    f" {'Category':<24}"
    f" {'PASS':>6}"
    f" {'REVIEW':>8}"
    f" {'Pass Rate':>12}"
    f" {'Avg Confidence':>16}"
)
print(_summary_rule("-"))

for category in test_suite.keys():
    category_results = [r for r in test_results if r["category"] == category]
    category_total = len(category_results)
    category_pass = sum(1 for r in category_results if r["is_confident"])
    category_review = category_total - category_pass
    category_avg_conf = (
        sum(r["confidence"] for r in category_results) / category_total
        if category_total
        else 0.0
    )
    category_pass_rate = (category_pass / category_total) if category_total else 0.0

    print(
        f" {category.upper().replace('_', ' '):<24}"
        f" {category_pass:>6}"
        f" {category_review:>8}"
        f" {_fmt_pct(category_pass_rate):>12}"
        f" {_fmt_pct(category_avg_conf):>16}"
    )

print(_summary_rule("="))

---

## Part 8: Production Performance Review

This section documents observed system behavior, operational strengths, and forward-looking engineering improvements based on evaluation outcomes.

### 8.1 Performance Analysis

**Highest-performing query classes**

The assistant performed most consistently on Simple Facts and Detailed Explanations where the target answer existed as a direct span in retrieved context. In these scenarios, retrieval alignment remained strong and the extractive model reliably selected semantically precise token boundaries.

**Lower-performing query classes**

Complex Comparison prompts were the primary stress case. These queries often require synthesis across multiple context regions, while an extractive model is optimized for selecting one contiguous answer span. As a result, response quality can degrade when correct output depends on multi-step aggregation or cross-paragraph reasoning.

**Out-of-bounds behavior and guardrail effectiveness**

Out-of-bounds prompts generally triggered the fallback path as expected. The confidence gate correctly suppressed direct answering when either retrieval distance or span-level certainty fell below threshold. This behavior reduced hallucination risk and made uncertainty explicit to end users.

### 8.2 Strengths and Limitations

**Production strengths**

1. Grounded response policy: The dual-gated confidence mechanism provides a deterministic reliability boundary and prevents unsupported answers from being surfaced.
2. Retrieval traceability: Source-aware metadata and explicit source reporting improve auditability for support and operations teams.
3. Deterministic extractive control: Manual token-span logit handling enables transparent answer construction and easier debugging than opaque generation-only workflows.

**Current limitations**

1. Limited multi-hop reasoning: Single-span extraction constrains performance when prompts require synthesis across multiple documents.
2. Retrieval dependency: Final answer quality remains tightly coupled to top-k retrieval relevance and chunk granularity.
3. Threshold sensitivity: Confidence calibration can drift by query style and context length, requiring periodic tuning for stable production behavior.

### 8.3 Engineering Improvement Roadmap

**Priority improvements for production hardening**

1. Add reranking after retrieval: Introduce a cross-encoder reranker to improve context ordering before extractive scoring.
2. Implement hierarchical chunking: Use parent-child chunk indexing to improve recall while preserving sufficient local context for answer extraction.
3. Expand observability and calibration: Track confidence distributions by query class and run scheduled threshold recalibration to maintain consistent fallback behavior over time.
4. Introduce hybrid answer mode: Combine extractive validation with a constrained generative layer for comparison-heavy prompts while preserving confidence gating as the final policy control.

---

## Part 9: Demo Showcase

Create a polished demo of your assistant in action.

In [ ]:
# ============================================================
# Create a demo showcasing your assistant
# ============================================================

def run_demo(assistant, demo_questions: List[str]):
    """
    Run a polished demo of the RAG assistant.
    """
    print("\n" + "="*70)
    print(f"{PROJECT_INFO['topic']} - Knowledge Assistant Demo")
    print(f"   {PROJECT_INFO['description']}")
    print("="*70)

    for i, question in enumerate(demo_questions, 1):
        print(f"\n{'─'*70}")
        print(f"User Question {i}:")
        print(f"   \"{question}\"")
        print()

        result = assistant.ask(question)

        print(f"Assistant Response:")
        print(f"   {result['answer']}")
        print()
        print(f"   Sources: {', '.join(result['sources'][:2])}")
        print(f"   Confidence: {result['confidence']:.1%}")

    print(f"\n{'='*70}")
    print("Demo complete!")
    print("="*70)


# Select 5 of your best questions for the demo
demo_questions = [
    # TODO: Choose 5 questions that showcase your system well
    "Demo question 1",
    "Demo question 2",
    "Demo question 3",
    "Demo question 4",
    "Demo question 5",
]

run_demo(assistant, demo_questions)

---

## System Features & Engineering Competencies

This delivery represents a production-oriented RAG system implementation with clear reliability, observability, and retrieval quality controls.

1. Knowledge Domain Engineering: Designed and curated a multi-document domain corpus with structured metadata to support deterministic retrieval and source traceability.
2. Context Segmentation Architecture: Implemented sentence-aware, overlap-based chunking to preserve semantic continuity while improving vector retrieval precision.
3. Vector Search Infrastructure: Built and populated a ChromaDB-backed semantic index for low-latency nearest-neighbor retrieval across embedded document segments.
4. Controlled Extractive Inference: Implemented manual token-span boundary selection from start/end logits to enforce transparent, auditable answer extraction behavior.
5. Dual-Gated Reliability Guardrails: Deployed confidence fusion across retrieval alignment and extractive span certainty, with fallback enforcement for low-confidence states.
6. Evaluation and Reporting Framework: Executed multi-category benchmark coverage (fact lookup, detailed explanation, complex comparison, out-of-bounds) with structured dashboard summaries.
7. Production Documentation Discipline: Captured architecture, trade-offs, and operational behavior in implementation notes to support maintainability and external technical review.